In [4]:
import math as m
import numpy as np
import random 
import stable_baselines3
import torch
import tensorflow as tf
import random 

model = tf.keras.Sequential([
    tf.keras.layers.Dense(4,input_shape=(16,))
])

x = np.random.rand(1,16)

print(model(x))

tf.Tensor([[-0.8375909  -0.5036671  -0.42239115  0.03344543]], shape=(1, 4), dtype=float32)


In [5]:
x= torch.rand(5,3)
x

tensor([[0.7581, 0.7286, 0.8049],
        [0.4628, 0.7437, 0.6434],
        [0.2034, 0.0092, 0.5725],
        [0.7786, 0.6501, 0.5279],
        [0.6355, 0.8718, 0.1946]])

In [6]:
# State / Env
class Box_in_Chessboard :
    def __init__(
            self,
            name,
            xy:tuple[int,int],
            color:str, 
            symbol:str=None, 
            dirlist : list = None, 
            reward_fn = None
        ):

        # this part describes the box itself
        self.name=name
        self.coordinate=xy
        self.color=color
        self.reward=0
        
        self.reward_fn = reward_fn if reward_fn != None else self.default_rewardfn
        
        self.symbol= symbol
        #this one is about the surrounding of the box
        self.neighbours={
            "up":dirlist[0],
            "down":dirlist[1],
            "right":dirlist[2],
            "left":dirlist[3],
            "stand":self
        }

    def default_rewardfn(self):
        if self.symbol == "X":
            self.reward = -100
        elif self.symbol == "O":
            self.reward = 100
        else:
            self.reward = -10

        return self.reward

    def reward_(self):
        return  self.reward_fn()

    def next_state(self,action):
        if self.neighbours[action] is None:
            return self
        return self.neighbours[action] 
    
    # 🔥 KEY FIX: equality based on identity of state meaning
    def __eq__(self, other):
        return isinstance(other, Box_in_Chessboard) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

In [7]:
#the state _ vector
# the box in chessboard in not actually a state but describing a specific element of the environment ,mixed with the concept of state in some aspect :
def state_to_vector(state):

    color = 0 if state.color == "white" else 1

    if state.symbol == "X":
        symbol = -1
    elif state.symbol == "O":
        symbol = 1
    else:
        symbol = 0

    return np.array([
        state.coordinate[0],
        state.coordinate[1],
        color,
        symbol
    ], dtype=np.float32)

def state_to_vectory(state):
    def manhattan_goal(state:Box_in_Chessboard):
        x,y = state.coordinate
        return abs(x-0) + abs(y-2)

    def manhattan_trap(state:Box_in_Chessboard):
        x,y = state.coordinate
        return abs(x-1) + abs(y-1)
    
    return np.array([
        manhattan_goal(state=state),
        manhattan_trap(state=state)
    ], dtype=np.float32)

In [8]:
#set up
s1 = Box_in_Chessboard("s1",(0,0),"white","X",[None,None,None,None])
s2 = Box_in_Chessboard("s2",(2,0),"white","O",[None,None,None,None])
s3 = Box_in_Chessboard("s3",(4,0),"white","X",[None,None,None,None])
s4 = Box_in_Chessboard("s4",(0,1),"white",None,[None,None,None,None])
s5 = Box_in_Chessboard("s5",(1,1),"grey",None,[None,None,None,None])
s6 = Box_in_Chessboard("s6",(2,1),"white",None,[None,None,None,None])
s7 = Box_in_Chessboard("s7",(3,1),"grey",None,[None,None,None,None])
s8 = Box_in_Chessboard("s8",(4,1),"white",None,[None,None,None,None])

#s0 = Box_in_Chessboard("empty box")

#define
s1.neighbours = {"up": s4,   "down": None, "right": None, "left": None, "stand": s1}
s2.neighbours = {"up": s6,   "down": None, "right": None, "left": None, "stand": s2}
s3.neighbours = {"up": s8,   "down": None, "right": None, "left": None, "stand": s3}
s4.neighbours = {"up": None, "down": s1,   "right": s5,   "left": None, "stand": s4}
s5.neighbours = {"up": None, "down": None, "right": s6,   "left": s4,   "stand": s5}
s6.neighbours = {"up": None, "down": s2,   "right": s7,   "left": s5,   "stand": s6}
s7.neighbours = {"up": None, "down": None, "right": s8,   "left": s6,   "stand": s7}
s8.neighbours = {"up": None, "down": s3,   "right": None, "left": s7,   "stand": s8}

#



States=[s1,s2,s3,s4,s5,s6,s7,s8]
Actions=["up", "right","down","left"]




In [9]:
# Env (organizing the puzzle of state into env )
class Env:

    @property
    def actions(self) -> list:
        """All actions available in this environment."""
        raise NotImplementedError

    @property
    def states(self) -> list:
        """All states in this environment."""
        raise NotImplementedError

    def reset(self):
        """Start a new episode. Return the initial state."""
        raise NotImplementedError

    def step(self, action):
        """
        Apply action to the current state.
        Return (next_state, reward, done).
        """
        raise NotImplementedError
    
class Chessboard(Env):
    def __init__(self,
                 states: list[Box_in_Chessboard],
                 actions:list[str],
                 start: Box_in_Chessboard):
        self._states = states
        self._actions = actions
        self.start = start
        self.current = start # the current state

    @property
    def states(self):
        return self._states
    @property
    def actions(self):
        return self._actions
    
    # methode

    def reset(self):
        self.current = self.start
        return self.current
    
    def step(self, action):
        """
        Apply action to the current state.
 
        Calls next_state()  → your Box_in_Chessboard transition logic
        Calls reward_()     → your Box_in_Chessboard reward logic
 
        Returns:
            next_state  — the Box_in_Chessboard we landed on
            reward      — float reward for this transition
            done        — True if episode should end
        """

        next_state = self.current.next_state(action)
        next_state.reward_()
        reward = next_state.reward
        done = next_state.symbol in ("X","O")

        self.current = next_state 

        return {
            "state":self.current, 
            "action" : action, 
            "reward" : reward, 
            "next_state":next_state , 
            "done":done
            }
    
Board_Skript = Chessboard(States,Actions,s4)


In [10]:
# class Policy:
#     """
#     A policy maps states to actions.
#     It can be deterministic or stochastic.
#     It can be a Q-table, a probability table, a neural net, anything.
#     The strategy writes into it. The agent reads from it.
#     """

#     def select_action(self, state, actions) -> str:
#         """
#         Given a state, return an action.
#         Deterministic  → always the same action for a given state.
#         Stochastic     → sample from a distribution over actions.
#         """
#         raise NotImplementedError

#     def update(self, **kwargs):
#         """
#         Absorb whatever the strategy learned.start = 
#         kwargs keeps this open — every strategy passes what it needs:
#           Q-learning  → update(state, action, value)
#           Policy grad → update(state, action, probability)
#           Actor-critic→ update(state, action, advantage, value)
#         """
#         raise NotImplementedError

#     def snapshot(self) -> dict:
#         """Return a serializable copy of the current policy state."""
#         raise NotImplementedError
    

# class Stocha_Policy(Policy):

#     def __init__(self,States:list[Box_in_Chessboard],Actions:list):
#         self.states = States
#         self.actions = Actions
#         self.pi = {}
    

#     def pi_0(self):
#         pi = {}

#         for s in self.states:
#             a = random.choice(self.actions)
#             n = self.actions.index(a)
#             if n % 2 != 0:
#                 n -= 1

#             pi[s.name] = {
#                 a: 0.8,
#                 self.actions[(n + 2) % 4]: 0.1,
#                 self.actions[(n + 3) % 4]: 0.1,
#             }
        
#         self.pi = pi
#     def select_action(self,state):
#         dic = self.pi[state.name]
#         keys = list( dic.keys() )
#         values = list( dic.values() )

#         action = random.choices(keys, weights=values, k=1)[0]
#         return action
    

# pi = Stocha_Policy(States ,Actions)
# pi.pi_0()

# f" {pi.pi} "

In [11]:
# phi(s) = [phi_1(s),phi_2(s)] = [dist_goal, dist_trap] = [ Manhattan(agent→goal), Manhattan(agent→trap) ]
# R(s,a) = +10 (if agent lands on goal cell) -5 (if agent lands on trap cell) -1 (for each movement)

# phi(s) = [phi_1(s),phi_2(s)] = [dist_goal, dist_trap] = [ Manhattan(agent→goal), Manhattan(agent→trap) ]
# R(s,a) = +10 (if agent lands on goal cell) -5 (if agent lands on trap cell) -1 (for each movement)
import math as m

class FeatureFunction:
    def __init__(self):
        self.buffer = []

    def add(self, *fn):
        for f in fn:
            self.buffer.append(f)

    def compute(self, state, action=None):
        current_state = state.next_state(action) if action is not None else state
        return np.array([fn(current_state) for fn in self.buffer], dtype=float)

    def phi_sa(self, state, action, stocha, rule_fn):
        if stocha:
            return rule_fn(self, state, action)
        return self.compute(state.next_state(action))


def ruler_Fn(x:FeatureFunction, state, action):
    n  = Actions.index(action)
    a1 = Actions[(n + 1) % 4]
    a2 = Actions[(n + 3) % 4]
    return 0.8 * x.compute(state.next_state(action)) + \
           0.1 * (x.compute(state.next_state(a1)) + x.compute(state.next_state(a2)))


# phi_list = []
# for s in States2:
#     phi_list.append(phi.compute(s,None))
# phi_list[0]@np.array([12,7])

In [12]:
from abc import ABC, abstractmethod

class Policy(ABC):
    @abstractmethod
    def select_action(self, state): pass
    @abstractmethod
    def action_distribution(self, state): pass
    @abstractmethod
    def snapshot(self): pass

class DeterministicPolicy(Policy):
    def __init__(self):
        self.mapping = {}
    def select_action(self, state):        return self.mapping[state]
    def action_distribution(self, state):  return {self.mapping[state]: 1.0}
    def snapshot(self):                    return self.mapping.copy()

class StochasticPolicy(Policy):
    def __init__(self):
        self.pi = {}
    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]
    def action_distribution(self, state):  return self.pi[state]
    def snapshot(self):                    return self.pi.copy()

class Policy_theta_softmax(Policy):
    def __init__(self, phi:FeatureFunction, theta):
        self.pi      = {}
        self.weights = theta
        self.phi     = phi

    def _zeller(self, state, action, Actions): #
        n  = Actions.index(action)
        a1 = Actions[(n + 1) % 4]
        a2 = Actions[(n + 3) % 4]
        phi_sa = 0.8 * self.phi.compute(state.next_state(action)) + \
                 0.1 * (self.phi.compute(state.next_state(a1)) +
                        self.phi.compute(state.next_state(a2)))
        h = (self.weights @ phi_sa).item()
        return np.exp(h)

    def calculate_all_probabilities(self, state, Actions):
        # compute each zeller once, reuse for denominator
        zellers = {a: self._zeller(state, a, Actions) for a in Actions}
        N = sum(zellers.values())
        self.pi[state] = {a: float(z / N) for a, z in zellers.items()}

    def select_action(self, state):
        actions = list(self.pi[state].keys())
        probs   = list(self.pi[state].values())
        return random.choices(actions, probs)[0]

    def action_distribution(self, state):  return self.pi.get(state, {})
    def snapshot(self):                    return self.pi.copy()

class NeuralPolicy(Policy):
    def __init__(self, network):
        self.network = network
    def select_action(self, state):
        probs = self.network(state)
        return 0  # placeholder
    def action_distribution(self, state):  return self.network(state)
    def snapshot(self):                    return self.network.state_dict()

In [13]:
import types
# def special_action(self):
#     return f"Special behavior for this instance: {self.pi}"

# stoch_policy = StochasticPolicy()

# stoch_policy.special_action = types.MethodType(special_action, stoch_policy)

pi_0 = StochasticPolicy()
Actions
for s in States:
    print(s.name)
def initialize_the_dih(States,Actions):
    dic = {}
    for s in States:
        a = random.choice(Actions)
        n = Actions.index(a)
        #if n % 2 != 0:n -= 1
        dic[s] = {
            a : 0.8,
            Actions[(n - 1)%4]:0.1,
            Actions[(n + 2)%4]:0.1,
            Actions[(n + 1)%4]:0.0
        }
    
    return dic

pi_0.pi = initialize_the_dih(States,Actions)
pi_0.snapshot()
pi_0.action_distribution(s1)


s1
s2
s3
s4
s5
s6
s7
s8


{'left': 0.8, 'down': 0.1, 'right': 0.1, 'up': 0.0}

In [14]:
from abc import ABC, abstractmethod

class LearningStrategy(ABC):
    @abstractmethod
    def update(self, policy, experience): pass
    @abstractmethod
    def snapshot(self): pass

class QLearning(LearningStrategy):
    def __init__(self, actions, alpha=0.1, gamma=0.9):
        self.alpha   = alpha
        self.gamma   = gamma
        self.actions = actions
        self.Q       = {}

    def _ensure(self, s):
        if s not in self.Q:
            self.Q[s] = {a: 0.0 for a in self.actions}

    def update(self, policy, exp):
        s, a, r, next_s, done = exp["state"], exp["action"], exp["reward"], exp["next_state"], exp["done"]
        self._ensure(s)
        self._ensure(next_s)
        target = r if done else r + self.gamma * max(self.Q[next_s].values())
        self.Q[s][a] += self.alpha * (target - self.Q[s][a])
        policy.mapping[s] = max(self.Q[s], key=self.Q[s].get)

    def snapshot(self):
        return self.Q.copy()

class Reinforce(LearningStrategy):
    def __init__(self, lr=0.1):
        self.lr = lr
    def update(self, policy, state_vec, action_index, reward):
        probs = policy.forward(state_vec)
        for i in range(policy.W.shape[1]):
            policy.W[0, i] += self.lr * reward * ((1 if i == action_index else 0) - probs[i]) * state_vec[0]
    def snapshot(self): return {}

class PPO(LearningStrategy):
    def __init__(self, optimizer, clip_epsilon=0.2):
        self.optimizer    = optimizer
        self.clip_epsilon = clip_epsilon
    def update(self, policy, batch): pass  # implement later
    def snapshot(self): return {"clip_epsilon": self.clip_epsilon}

In [15]:
# ════════════════════════════════════════════════════════════════════
#  BRICK 4 — AGENT
#  The self. Owns the policy (permanent). Borrows the strategy (swappable).
#  Contains no learning logic — purely coordinates the other bricks.
# ════════════════════════════════════════════════════════════════════
    
class Agent:

    def __init__(self, strategy: LearningStrategy, policy: Policy):
        self.policy   = policy    # permanent — never replaced
        self.strategy = strategy  # swappable — plug any algorithm in
        self.memory   = []        # full experience trace

    def act(self, state) -> str:
        """Ask the strategy what to do, passing the policy as context."""
        return self.policy.select_action(state)

    def learn(self, experience:dict): # experience = {"state": "A","action": "right","reward": 1,"next_state": "B","done": False}
        """Tell the strategy what happened; it writes into the policy."""
        state,action,reward ,next_state,done = experience.values()

        self.strategy.update(self.policy, experience)
        self.memory.append((state.name, action, reward, next_state.name, done))

    def swap_strategy(self, new_strategy: LearningStrategy):
        """
        Replace the learning algorithm.
        The policy — and everything it has learned — is untouched.
        """
        self.strategy = new_strategy

In [16]:
class Agent2:
    def __init__(self,state:Box_in_Chessboard,actions,goal,policy):
        self.state = state
        self.goal = goal # condition of train loop
        self.actions = actions # use it for Q(s,a)
        #------#
        self.policy = policy
        self.strategy = None
        self.memory = []

    def act(self): # follows the policy
        action = self.policy.select_action(self.state)
        next_state = self.state.next_state(action)
        self.state = next_state
        self.state.reward_()
    
    def learn(self):pass

    def swap_strategy(self,new_strategy):
        self.strategy = new_strategy

#agent_cooper = Agent(s4,Actions,s2,pi)

In [17]:
s1 = Box_in_Chessboard("s1",(2,0),"white",None,[None,None,None,None])
s2 = Box_in_Chessboard("s2",(2,1),"white",None,[None,None,None,None])
s3 = Box_in_Chessboard("s3",(2,2),"white",None,[None,None,None,None])
s4 = Box_in_Chessboard("s4",(1,0),"white",None,[None,None,None,None])
s5 = Box_in_Chessboard("s5",(1,1),"white","X",[None,None,None,None])
s6 = Box_in_Chessboard("s6",(1,2),"white",None,[None,None,None,None])
s7 = Box_in_Chessboard("s7",(0,0),"white",None,[None,None,None,None])
s8 = Box_in_Chessboard("s8",(0,1),"white",None,[None,None,None,None])
s9 = Box_in_Chessboard("s9",(0,2),"white","O",[None,None,None,None])

s1.neighbours = {"up": s4,   "down": None, "right": s2,   "left": None, "stand": s1}
s2.neighbours = {"up": s5,   "down": None, "right": s3,   "left": s1,   "stand": s2}
s3.neighbours = {"up": s6,   "down": None, "right": None,  "left": s2,   "stand": s3}
s4.neighbours = {"up": s7,   "down": s1,   "right": s5,   "left": None, "stand": s4}
s5.neighbours = {"up": s8,   "down": s2,   "right": s6,   "left": s4,   "stand": s5}  # ← fixed "down": s2 not s5
s6.neighbours = {"up": s9,   "down": s3,   "right": None,  "left": s5,   "stand": s6}
s7.neighbours = {"up": None, "down": s4,   "right": s8,   "left": None, "stand": s7}
s8.neighbours = {"up": None, "down": s5,   "right": s9,   "left": s7,   "stand": s8}
s9.neighbours = {"up": None, "down": s6,   "right": None,  "left": s8,   "stand": s9}

States2 = [s1,s2,s3,s4,s5,s6,s7,s8,s9]
Actions  = ["up","right","down","left"]

Claude_board = Chessboard(States2, Actions, s1)

In [18]:
class ValueFunction :
    def __init__(self,phi:FeatureFunction,theta:list):
        self.phi = phi
        self.Weights = theta # list of weights
        #self.n = n # if n=0 TD(0) target & n=inf means monte-carlo
        #self.policy= policy 
    
    def estimate(self,state): 
        phi_s = self.phi.compute(state)
        value = np.dot(phi_s, self.Weights)
        return float(f"{value:.2f}")
        #return float(np.dot(phi_s, self.Weights))
    def update(self, state , td_error:float , alpha:float = 0.01):
        phi_s = self.phi.compute(state,action=None)
        self.Weights += alpha*td_error*phi_s

#w1,w2 = theta.flatten()
#theta = np.random.randn(2,1)
#VF = ValueFunction(pi_0,phi,theta)

In [19]:
class SimpleLearn(LearningStrategy):
    def __init__(self, policy:Policy, VF:ValueFunction, n=0, gamma=0.9, alpha=0.01, beta=0.01):
        self.TD    = n
        self.pi    = policy
        self.VF    = VF
        self.gamma = gamma
        self.alpha = alpha
        self.beta  = beta

    def TD_error(self, s, s_next, r, done):
        target   = r if done else r + self.gamma * self.VF.estimate(s_next)
        td_error = target - self.VF.estimate(s)
        return td_error

    def Gradient(self, state, policy):
        phi = policy.phi
        dL  = phi.compute(state)
        for a in Actions:
            dL += self.pi.action_distribution(state).get(a, 0.0) * \
                  phi.phi_sa(state, a, stocha=True, rule_fn=ruler_Fn)
        return dL

    def update(self, policy, experience):
        s      = experience["state"]
        s_next = experience["next_state"]
        r      = experience["reward"]
        done   = experience["done"]           # ← fixed

        td_error = self.TD_error(s, s_next, r, done)

        phi_s = self.VF.phi.compute(s, action=None)
        self.VF.Weights -= self.alpha * td_error * phi_s

        dL_pi = self.Gradient(s, policy)
        policy.weights += self.beta * td_error * dL_pi

    def snapshot(self): return {}

class PPO(LearningStrategy):
    def __init__(self, optimizer, clip_epsilon=0.2):
        self.optimizer    = optimizer
        self.clip_epsilon = clip_epsilon
    
    def Clip(self):
        # we clip the limit 
        Rho = self.optimizer
        eps_max = 1 + self.clip_epsilon
        eps_min = 1 - self.clip_epsilon 
        if self.optimizer > eps_max: return eps_max
        elif self.optimizer < eps_min : return eps_min
        else : return Rho

    def Advantage(self,s,a):
        pass

    def update(self, policy, batch): pass  # implement later
    def snapshot(self): return {"clip_epsilon": self.clip_epsilon}
        

In [20]:
# --- setup ---
env = Claude_board
env.reset()

def manhattan_goal(state:Box_in_Chessboard):
    x,y = state.coordinate
    return abs(x-0) + abs(y-2)

def manhattan_trap(state:Box_in_Chessboard):
    x,y = state.coordinate
    return abs(x-1) + abs(y-1)

phi       = FeatureFunction()
phi.add(manhattan_goal, manhattan_trap)
theta_VF  = np.random.randn(2,)
VF        = ValueFunction(phi, theta_VF)

theta_pi  = np.random.randn(2,)
pi_1      = Policy_theta_softmax(phi, theta_pi)

for s in env.states:
    pi_1.calculate_all_probabilities(s, env.actions)  # ← fixed method name

Learning      = SimpleLearn(pi_1, VF)
agent_cooper  = Agent(Learning, pi_1)

T = 1000
env.reset()
for episode in range(T):
    s          = env.current
    a          = agent_cooper.act(s)
    experience = env.step(a)          # ← now returns dict
    agent_cooper.learn(experience)
    if experience["done"]:
        env.reset()

In [21]:
pi_snapdhot = agent_cooper.policy.snapshot()
for k,v in pi_snapdhot.items():
    print(f"{k.name} : \n{v}\n")
    #print(sum(v.values()))
    a=agent_cooper.policy.select_action(k)
    print(a)
    print(f"end of the state {k.name}")

#I want to vizualize through matplotlib how to the agent moves 

s1 : 
{'up': 0.04283674620271735, 'right': 0.04283674620271735, 'down': 0.4571632537972826, 'left': 0.4571632537972826}

left
end of the state s1
s2 : 
{'up': 0.011371528459909762, 'right': 0.0209465577383098, 'down': 0.12135947316771172, 'left': 0.8463224406340687}

left
end of the state s2
s3 : 
{'up': 0.022810455097554767, 'right': 0.15331543895683766, 'down': 0.24343823463261668, 'left': 0.5804358713129909}

left
end of the state s3
s4 : 
{'up': 0.0209465577383098, 'right': 0.011371528459909762, 'down': 0.8463224406340687, 'left': 0.12135947316771172}

down
end of the state s4
s5 : 
{'up': 0.012076159633066913, 'right': 0.012076159633066913, 'down': 0.48792384036693304, 'left': 0.48792384036693304}

left
end of the state s5
s6 : 
{'up': 0.01698745138006582, 'right': 0.06198496307873428, 'down': 0.6863591379425369, 'left': 0.23466844759866307}

down
end of the state s6
s7 : 
{'up': 0.15331543895683766, 'right': 0.022810455097554763, 'down': 0.5804358713129908, 'left': 0.243438234632

In [22]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

class GridVisualizer:
    def __init__(self, env: Chessboard, agent: Agent, transition_frames=12, frame_delay=0.03):
        self.env   = env
        self.agent = agent
        self.transition_frames = transition_frames
        self.frame_delay       = frame_delay

        xs = [s.coordinate[0] for s in env.states]
        ys = [s.coordinate[1] for s in env.states]
        self.x_min, self.x_max = min(xs), max(xs)
        self.y_min, self.y_max = min(ys), max(ys)

        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(6, 6))
        self.agent_marker = None
        self.agent_pos     = list(self.env.current.coordinate)  # current visual position (floats)

        self.step_button  = widgets.Button(description="Step ▶")
        self.reset_button = widgets.Button(description="Reset ⟲")
        self.log          = widgets.Output()
        self.plot_out     = widgets.Output()

        self.step_button.on_click(self._on_step)
        self.reset_button.on_click(self._on_reset)

        display(widgets.HBox([self.step_button, self.reset_button]))
        display(self.plot_out)
        display(self.log)

        self._draw_grid()
        self._draw_agent()
        self._render()

    def _color_for(self, state: Box_in_Chessboard):
        if state.symbol == "O": return "#8fd694"
        if state.symbol == "X": return "#e88080"
        return "#f0f0f0"

    def _draw_grid(self):
        self.ax.clear()

        for s in self.env.states:
            x, y = s.coordinate
            rect = patches.Rectangle(
                (x - 0.5, y - 0.5), 1, 1,
                facecolor=self._color_for(s),
                edgecolor="black", linewidth=1.2
            )
            self.ax.add_patch(rect)
            self.ax.text(x, y, s.name, ha="center", va="center", fontsize=9, zorder=4)

        drawn = set()
        for s in self.env.states:
            for direction, neighbour in s.neighbours.items():
                if neighbour is None or neighbour is s or direction == "stand":
                    continue
                pair = tuple(sorted([s.name, neighbour.name]))
                if pair in drawn:
                    continue
                drawn.add(pair)
                x1, y1 = s.coordinate
                x2, y2 = neighbour.coordinate
                self.ax.plot([x1, x2], [y1, y2], color="grey", linewidth=1, zorder=1)

        self.ax.set_xlim(self.x_min - 1, self.x_max + 1)
        # FIX: s1 bottom-left → normal (non-inverted) y-axis, low y at bottom
        self.ax.set_ylim(self.y_min - 1, self.y_max + 1)
        self.ax.set_aspect("equal", adjustable="box")
        self.ax.set_xticks([])
        self.ax.set_yticks([])

    def _draw_agent(self):
        x, y = self.agent_pos
        if self.agent_marker is not None:
            self.agent_marker.remove()
        self.agent_marker = patches.Circle((x, y), 0.25, facecolor="#3a7bd5", zorder=5)
        self.ax.add_patch(self.agent_marker)

    def _render(self):
        with self.plot_out:
            clear_output(wait=True)
            display(self.fig)

    def _animate_move(self, start, end):
        sx, sy = start
        ex, ey = end
        for i in range(1, self.transition_frames + 1):
            t = i / self.transition_frames
            self.agent_pos = [sx + (ex - sx) * t, sy + (ey - sy) * t]
            self._draw_agent()
            self._render()
            time.sleep(self.frame_delay)

    def _on_step(self, _):
        state      = self.env.current
        start_pos  = list(state.coordinate)
        action     = self.agent.act(state)
        experience = self.env.step(action)
        self.agent.learn(experience)

        end_pos = list(experience["next_state"].coordinate)
        self._animate_move(start_pos, end_pos)

        with self.log:
            print(f"{state.name} --{action}--> {experience['next_state'].name} "
                  f"| reward = {experience['reward']} | done = {experience['done']}")
            if experience["done"]:
                print("Episode finished — resetting\n")
                self.env.reset()
                self.agent_pos = list(self.env.current.coordinate)
                self._draw_agent()
                self._render()

    def _on_reset(self, _):
        self.env.reset()
        self.agent_pos = list(self.env.current.coordinate)
        self._draw_agent()
        self._render()
        with self.log:
            print("Environment reset\n")


Claude_board = Chessboard(States2, Actions, s1)
#viz = GridVisualizer(Claude_board, agent_cooper)

In [23]:
import numpy as np

def compute_gae(deltas, gamma=0.99, lam=0.95):
    """Compute GAE advantages from a list of TD residuals (deltas)"""
    advantages = torch.zeros(len(deltas))
    gae = torch.tensor(0.0)
    
    # Backward pass - this is the key
    for t in reversed(range(len(deltas))):
        gae = deltas[t] + gamma * lam * gae
        advantages[t] = gae
    
    return advantages

# Simulate adding steps one by one (as in your question)
deltas = []

print("Incremental GAE Update:\n")

for step in range(1, 7):
    # Simulate receiving a new step
    new_delta = round(np.random.uniform(-1.0, 2.0), 3)   # random for demo
    deltas.append(new_delta)
    
    print(f"After step {step} (new δ = {new_delta}):")
    
    # Recompute GAE using ALL data available so far
    advantages = compute_gae(deltas, gamma=0.99, lam=0.95)
    
    for i, adv in enumerate(advantages):
        print(f"   A[{i+1:2d}] = {adv:.3f}")
    print("-" * 50)

Incremental GAE Update:

After step 1 (new δ = 0.135):
   A[ 1] = 0.135
--------------------------------------------------
After step 2 (new δ = -0.427):
   A[ 1] = -0.267
   A[ 2] = -0.427
--------------------------------------------------
After step 3 (new δ = 1.85):
   A[ 1] = 1.370
   A[ 2] = 1.313
   A[ 3] = 1.850
--------------------------------------------------
After step 4 (new δ = 1.736):
   A[ 1] = 2.814
   A[ 2] = 2.848
   A[ 3] = 3.483
   A[ 4] = 1.736
--------------------------------------------------
After step 5 (new δ = -0.369):
   A[ 1] = 2.525
   A[ 2] = 2.542
   A[ 3] = 3.156
   A[ 4] = 1.389
   A[ 5] = -0.369
--------------------------------------------------
After step 6 (new δ = 1.234):
   A[ 1] = 3.433
   A[ 2] = 3.507
   A[ 3] = 4.183
   A[ 4] = 2.480
   A[ 5] = 0.792
   A[ 6] = 1.234
--------------------------------------------------


In [24]:
Claude_board.states[0].__dict__


{'name': 's1',
 'coordinate': (2, 0),
 'color': 'white',
 'reward': -10,
 'reward_fn': <bound method Box_in_Chessboard.default_rewardfn of <__main__.Box_in_Chessboard object at 0x71e504738fe0>>,
 'symbol': None,
 'neighbours': {'up': <__main__.Box_in_Chessboard at 0x71e3bf8ef8c0>,
  'down': None,
  'right': <__main__.Box_in_Chessboard at 0x71e3bf8efbc0>,
  'left': None,
  'stand': <__main__.Box_in_Chessboard at 0x71e504738fe0>}}

In [25]:
s_Vector = state_to_vector(Claude_board.states[0])
s_Vector

array([2., 0., 0., 0.], dtype=float32)

In [26]:
import torch
import torch.nn as nn
x = torch.rand([2,3])

print(x)
print(x.shape)
print(x.dtype)
print(x.device)

layer1 = nn.Linear(3,2)

print(layer1.weight)
print(layer1.bias)

class My_Module(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    def forward(self,x):
        return layer1(x)
    

model = My_Module()

y= model(x)
y
#15620033

tensor([[0.7078, 0.5684, 0.5327],
        [0.2444, 0.3069, 0.4158]])
torch.Size([2, 3])
torch.float32
cpu
Parameter containing:
tensor([[ 0.2077, -0.0472, -0.1797],
        [ 0.3566,  0.3775,  0.4359]], requires_grad=True)
Parameter containing:
tensor([-0.0855,  0.3273], requires_grad=True)


tensor([[-0.0610,  1.0264],
        [-0.1239,  0.7116]], grad_fn=<AddmmBackward0>)

In [27]:
Claude_board.__dict__

{'_states': [<__main__.Box_in_Chessboard at 0x71e504738fe0>,
 '_actions': ['up', 'right', 'down', 'left'],
 'start': <__main__.Box_in_Chessboard at 0x71e504738fe0>,
 'current': <__main__.Box_in_Chessboard at 0x71e504738fe0>}

In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ── BRICK 1: reusable block (your Block, kept exactly) ──────────────────
class Block(nn.Module):
    """A reusable residual-style chunk: linear → norm → relu"""
    def __init__(self, dim):
        super().__init__()
        self.fc   = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return F.relu(self.norm(self.fc(x)))  # ← fixed: F.relu() not nn.ReLU()


# ── BRICK 2: shared body ─────────────────────────────────────────────────
class SharedBody(nn.Module):
    """
    Encodes the state φ(s) into a shared representation.
    Both actor and critic read from this — they see the same features.
    Input:  φ(s) of shape (n_features,)
    Output: hidden representation of shape (hidden_dim,)
    """
    def __init__(self, n_features: int, hidden_dim: int = 64):
        super().__init__()
        self.input_layer = nn.Linear(n_features, hidden_dim)
        self.block1      = Block(hidden_dim)
        self.block2      = Block(hidden_dim)

        #self.network = nn.Sequential(....)

    def forward(self, x):
        x = F.relu(self.input_layer(x))
        x = self.block1(x)
        x = self.block2(x)
        return x


# ── BRICK 3: actor head ──────────────────────────────────────────────────
class ActorHead(nn.Module):
    """
    Takes shared body output → outputs a probability distribution over actions.
    π_θ(a|s) = softmax(W · h + b)
    """
    def __init__(self, hidden_dim: int, n_actions: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, n_actions)

    def forward(self, h):
        return F.softmax(self.head(h), dim=-1)   # shape: (n_actions,)


# ── BRICK 4: critic head ─────────────────────────────────────────────────
class CriticHead(nn.Module):
    """
    Takes shared body output → outputs a single scalar V(s).
    No activation — value can be any real number.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, h):
        return self.head(h).squeeze(-1)           # shape: scalar


# ── BRICK 5: full Actor-Critic network ───────────────────────────────────
class ActorCritic(nn.Module):
    """
    One network, two outputs:
        actor  → π_θ(a|s)   used for: action selection + policy loss
        critic → V_θ(s)     used for: GAE computation  + value loss
    """
    def __init__(self, n_features: int, n_actions: int, hidden_dim: int = 64):
        super().__init__()
        self.body   = SharedBody(n_features, hidden_dim)
        self.actor  = ActorHead(hidden_dim, n_actions)
        self.critic = CriticHead(hidden_dim)

    def forward(self, x):
        h     = self.body(x)
        probs = self.actor(h)    # π_θ(a|s)
        value = self.critic(h)   # V_θ(s)
        return probs, value

    def get_action(self, state_vec: np.ndarray):
        """
        Given a raw state vector, returns:
          action   — sampled from π_θ(a|s)
          log_prob — log π_θ(a|s), needed for PPO ratio
          value    — V_θ(s), stored in rollout buffer
        """
        x           = torch.tensor(state_vec, dtype=torch.float32)
        probs, value = self.forward(x)
        dist        = torch.distributions.Categorical(probs)
        action_idx  = dist.sample()
        log_prob    = dist.log_prob(action_idx)
        return action_idx.item(), log_prob, value


# ── BRICK 6: your custom loss (kept from your code) ──────────────────────
class MyLoss(nn.Module):
    """MSE loss — used for critic: (V(s) - y_t)²"""
    def forward(self, y_pred, y_true):
        return ((y_pred - y_true) ** 2).mean()
    


# ── quick sanity check ───────────────────────────────────────────────────
n_features = 2   # [dist_goal, dist_trap] — your phi(s) size
n_actions  = 4   # up, right, down, left

net = ActorCritic(n_features=n_features, n_actions=n_actions, hidden_dim=64)

dummy_state = torch.tensor([2.0, 1.0], dtype=torch.float32)
probs, value = net(dummy_state)

print(f"Action probs : {probs.detach().numpy()}")  # 4 numbers summing to 1
print(f"Value        : {value.item():.4f}")        # single scalar
print(f"Total params : {sum(p.numel() for p in net.parameters())}")

# act from a real numpy feature vector
state_vec = np.array([2.0, 1.0])
action_idx, log_prob, val = net.get_action(state_vec)
print(f"Sampled action index: {action_idx} → {Actions[action_idx]}")

Action probs : [0.19459622 0.2501475  0.3609619  0.19429435]
Value        : 0.0193
Total params : 9093
Sampled action index: 2 → down


In [29]:
class Loss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, *args, **kwargs):
        raise NotImplementedError

class Loss_Entropy(Loss):
    """L_H = -Σ p·log(p)  — encourages exploration"""
    def forward(self, probs: torch.Tensor) -> torch.Tensor:
        eps = 1e-8
        return -(probs * torch.log(probs + eps)).sum()

class Loss_Critic(Loss):
    """L_V = (V(s) - y)²  — makes critic predict returns accurately"""
    def forward(self, v_pred: torch.Tensor, v_target: torch.Tensor) -> torch.Tensor:
        return ((v_pred - v_target) ** 2).mean()

class Loss_Actor(Loss):
    """L_pi = -ρ·A  — PPO clipped policy gradient"""
    def __init__(self, clip_epsilon=0.2):
        super().__init__()
        self.eps = clip_epsilon

    def forward(self, log_prob_new: torch.Tensor,
                      log_prob_old: torch.Tensor,
                      advantage:   torch.Tensor) -> torch.Tensor:
        # ratio ρ = π_new / π_old  via log difference (numerically stable)
        rho     = torch.exp(log_prob_new - log_prob_old)
        clipped = torch.clamp(rho, 1 - self.eps, 1 + self.eps)
        # take the pessimistic (minimum) of clipped and unclipped
        L_clip  = torch.min(rho * advantage, clipped * advantage)
        return -L_clip.mean()   # negative because we MAXIMIZE, but optimizer MINIMIZES

class Loss_PPO(Loss):
    """Total PPO loss = L_actor + cv·L_critic - ce·L_entropy"""
    def __init__(self, cv=0.5, ce=0.01, clip_epsilon=0.2):
        super().__init__()
        self.cv    = cv
        self.ce    = ce
        self.L_pi  = Loss_Actor(clip_epsilon)
        self.L_v   = Loss_Critic()
        self.L_h   = Loss_Entropy()

    def forward(self,
                log_prob_new: torch.Tensor,
                log_prob_old: torch.Tensor,
                advantage:    torch.Tensor,
                v_pred:       torch.Tensor,
                v_target:     torch.Tensor,
                probs:        torch.Tensor) -> torch.Tensor:

        actor_loss  = self.L_pi(log_prob_new, log_prob_old, advantage)
        critic_loss = self.L_v(v_pred, v_target)
        entropy     = self.L_h(probs)

        return actor_loss + self.cv * critic_loss - self.ce * entropy

In [30]:
# loss_fn = Loss_PPO(cv=0.5, ce=0.01)

# loss = loss_fn(
#     rho=ratio,
#     advantage=advantages,
#     value_pred=values,
#     value_target=returns,
#     probs=action_probs
# )

# loss.backward()
# optimizer.step()

In [31]:
class Policy_PPO(Policy):
    def __init__(self,network):
        super().__init__()
        self.pi ={}
        self.net = network
        #self.log = []
        #self.value = {} # to store the values for each state - defined externally
        # not the primary fn of pi but we can save some computation energy
         
    def select_action(self, state):
        x = state_to_vectory(state)
        a_idx , log_prob , value = self.net.get_action(x) # see what we can do with the value
        #self.log.append(log_prob)
        #self.value[state] = value
        return Actions[a_idx] , log_prob, value

    def action_distribution(self, state):
        x = torch.tensor(state_to_vectory(state),dtype= torch.float32)
        probs, value = self.net.forward(x)
        
        return self.net(x)
    def snapshot(self): return self.net.state_dict()
        

pi = Policy_PPO(ActorCritic(n_features= 2, n_actions=n_actions, hidden_dim=64)) 
pi.select_action(s6)

('left',
 tensor(-1.2015, grad_fn=<SqueezeBackward1>),
 tensor(-0.9255, grad_fn=<SqueezeBackward1>))

In [32]:
print(pi.action_distribution(s6))
#pi.snapshot()
x = torch.tensor(state_to_vectory(s6.next_state("up")))
_,v =pi.net(x)
v

(tensor([0.2583, 0.2305, 0.2104, 0.3008], grad_fn=<SoftmaxBackward0>), tensor(-0.9255, grad_fn=<SqueezeBackward1>))


tensor(-0.8883, grad_fn=<SqueezeBackward1>)

In [33]:
# --- rollout ---
s = s1
deltas       = []
log_probs_old = []
states_visited = []
actions_taken  = []
values_visited = []

for t in range(10-1):
    a,log_prob,v_s = pi.select_action(s)
    n_s = s.next_state(a)
    n_s.reward_()
    r = float(n_s.reward)
    x = torch.tensor(state_to_vectory(n_s))
    _,v_ns = pi.net(x) #or x

    delta = r + 0.99*v_ns.detach() -v_s
    
    deltas.append(delta)
    states_visited.append(s)
    actions_taken.append(a)
    values_visited.append(v_s)
    log_probs_old.append(log_prob.detach())

    s = n_s
    if n_s.symbol in ("X","O"): break
    

deltas



[tensor(-9.9938, grad_fn=<SubBackward0>),
 tensor(-10.0919, grad_fn=<SubBackward0>),
 tensor(-9.9166, grad_fn=<SubBackward0>),
 tensor(-9.9935, grad_fn=<SubBackward0>),
 tensor(-10.2706, grad_fn=<SubBackward0>),
 tensor(100.0460, grad_fn=<SubBackward0>)]

In [34]:
log_probs_new = []
values_new = []
new_probs_all = []

A = compute_gae(deltas=deltas)

v_tensor = torch.stack(values_visited)
returns = A + v_tensor

for s,a in zip(states_visited,actions_taken):
    x = torch.tensor(state_to_vectory(s))
    probs , v = pi.net(x)
    dist = torch.distributions.Categorical(probs)
    a_idx = Actions.index(a)
    log_prob = dist.log_prob(torch.tensor(a_idx))

    log_probs_new.append(log_prob)
    values_new.append(v)
    new_probs_all.append(probs)


#log_probs_old = torch.stack(log_probs_old)
log_probs_new = torch.stack(log_probs_new)
log_probs_old = torch.stack(log_probs_old)
V_pred = torch.stack(values_new)
probs_torch = torch.stack(new_probs_all)




In [35]:
optimizer = torch.optim.Adam(pi.net.parameters(),lr = 3e-4)
loss_fn = Loss_PPO()



loss= loss_fn(
    log_probs_new,
    log_probs_old,
    A,
    V_pred,
    returns,
    probs_torch
)

optimizer.zero_grad()

print(loss)
print(loss.shape)

loss.backward()


: 

In [ ]:
print(loss_fn.L_pi(log_probs_new, log_probs_old, A))
print(loss_fn.L_v(V_pred, returns))
print(loss_fn.L_h(probs_torch))
print(A)
print(returns)
print(V_pred)

tensor(112.3901, grad_fn=<NegBackward0>)
tensor(12692.1006, grad_fn=<MeanBackward0>)
tensor(10.7009, grad_fn=<NegBackward0>)
tensor([-123.5274, -120.7845, -117.7993, -114.5571, -111.1786, -107.6544,
        -103.8385,  -99.7812], grad_fn=<CopySlices>)
tensor([-124.1644, -121.3568, -118.3716, -115.1941, -111.8156, -108.2268,
        -104.4109, -100.3536], grad_fn=<AddBackward0>)
tensor([-0.6370, -0.5723, -0.5723, -0.6370, -0.6370, -0.5723, -0.5723, -0.5723],
       grad_fn=<StackBackward0>)


In [1]:
pip install torch


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
